In [8]:
import pandas as pd

###Nhập dataset

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
file_path = ('data/Dataset.csv')
dataset = pd.read_csv(file_path)
print(f"Đã tải {len(dataset)} hàng lên")


Đã tải 50000 hàng lên


###Tiền xử lí

####Loại bỏ từ dừng (stop words)

In [7]:
import nltk
from nltk.tokenize.toktok import ToktokTokenizer

ModuleNotFoundError: No module named 'nltk'

In [2]:
import nltk
nltk.download('stopwords')
tokenizer = ToktokTokenizer()
#lấy danh sách các từ dừng đã được định nghĩa sẵn trong tiếng Anh từ kho corpus
#nhằm giảm nhiễu, giúp tập trung vào các từ khóa thực sự quan trọng và giảm kích
#thước dữ liệu
stopword_list = nltk.corpus.stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
def remove_stopwords(text, is_lower_case=False):
    tokens = tokenizer.tokenize(text)
    tokens = [token.strip() for token in tokens]
    if is_lower_case:
        filtered_tokens = [token for token in tokens if token not in stopword_list]
    else:
        filtered_tokens = [token for token in tokens if token.lower() not in stopword_list]
    filtered_text = ' '.join(filtered_tokens)
    return filtered_text


In [4]:
dataset['review'] = dataset['review'].apply(remove_stopwords)

NameError: name 'dataset' is not defined

####Khử nhiễu văn bản

In [ ]:
from bs4 import BeautifulSoup

In [ ]:
#loại bỏ các thẻ HTML như <br> ra khỏi văn bản (vì dữ liệu trên IMDB là các đoạn mã HTML)
def strip_html(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

In [ ]:
import re #Regex

In [ ]:
#loại bỏ các ghi chú trong dấu ngoặc vuông và dấu ngoặc vuông
def remove_between_square_brackets(text):
    return re.sub('\[[^]]*\]', '', text)

In [ ]:
#hàm khử nhiễu dùng 2 hàm vừa định nghĩa
def denoise_text(text):
    text = strip_html(text)
    text = remove_between_square_brackets(text)
    return text

In [ ]:
#khử nhiễu cho dataset
dataset['review'] = dataset['review'].apply(denoise_text)

####Loại bỏ ký tự đặc biệt

In [ ]:
#hàm loại bỏ các ký tự đặc biệt (ký tự khác chữ, số, khoảng trắng và xuống dòng)
def remove_special_characters(text):
    pattern=r'[^a-zA-z0-9\s]'
    text=re.sub(pattern,'',text)
    return text

In [ ]:
#áp dụng cho dataset
dataset['review'] = dataset['review'].apply(remove_special_characters)

####Trích gốc từ tiếng Anh

In [ ]:
from nltk.stem.porter import PorterStemmer

In [ ]:
def simple_stemmer(text):
    ps = nltk.porter.PorterStemmer()
    text = ' '.join([ps.stem(word) for word in text.split()])
    return text

In [ ]:
dataset['review'] = dataset['review'].apply(simple_stemmer)

###Tách dataset thành tập train và test




In [ ]:
train_reviews = dataset.review[:40000]
train_sentiments = dataset.sentiment[:40000]
print(train_reviews.shape, train_sentiments.shape)

(40000,) (40000,)


In [ ]:
test_reviews = dataset.review[40000:]
test_sentiments = dataset.sentiment[40000:]

###Trích xuất đặc trưng


####Bag of words (BOW)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
cv=CountVectorizer(min_df=5, max_df=0.8, binary=False, ngram_range=(1,3))
#train
cv_train_reviews = cv.fit_transform(train_reviews)
#test
cv_test_reviews = cv.transform(test_reviews)

print('Kích thước tập cv_train_reviews:',cv_train_reviews.shape)
print('Kích thước tập cv_test_reviews:',cv_test_reviews.shape)

Kích thước tập cv_train_reviews: (40000, 390067)
Kích thước tập cv_test_reviews: (10000, 390067)


####Term Frequency - Inverse Document Frequency (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tv = TfidfVectorizer(min_df=5, max_df=0.8, use_idf=True, ngram_range=(1,3))

tv_train_reviews = tv.fit_transform(train_reviews)

tv_test_reviews = tv.transform(test_reviews)

print('Kích thước tập tv_train_reviews:',tv_train_reviews.shape)
print('Kích thước tập tv_test_reviews:',tv_test_reviews.shape)

Kích thước tập tv_train_reviews: (40000, 390067)
Kích thước tập tv_test_reviews: (10000, 390067)


###Gán nhãn nhị phân cho cột sentiment (positive và negative)




In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
lb = LabelBinarizer()
#chuyển đổi dữ liệu sentiment
sentiment_data=lb.fit_transform(dataset['sentiment'])
print(sentiment_data.shape)

(50000, 1)


In [ ]:
train_sentiments = sentiment_data[:40000]
test_sentiments = sentiment_data[40000:]

print(train_sentiments)
print(test_sentiments)

[[1]
 [1]
 [1]
 ...
 [1]
 [0]
 [0]]
[[0]
 [0]
 [0]
 ...
 [0]
 [0]
 [0]]


[[0]
 [0]
 [0]
 ...
 [0]
 [0]
 [0]]


##Xây dựng mô hình dự đoán Logistic Regression và Multinomial Naive Bayes

###Logistic Regression

#####Huấn luyện mô hình

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

In [ ]:
lr_bow_model = LogisticRegression(penalty='l2',
                        max_iter=500,
                        C=1,
                        random_state=42)

lr_tfidf_model = LogisticRegression(penalty='l2',
                        max_iter=500,
                        C=1,
                        random_state=42)

nb_bow_model = MultinomialNB()
nb_tfidf_model = MultinomialNB()

#Bow
lr_bow = lr_bow_model.fit(cv_train_reviews, train_sentiments)
print(lr_bow)
nb_bow = nb_bow_model.fit(cv_train_reviews, train_sentiments)
print(nb_bow)

#TFIDF
lr_tfidf = lr_tfidf_model.fit(tv_train_reviews, train_sentiments)
print(lr_tfidf)
nb_tfidf = nb_tfidf_model.fit(tv_train_reviews, train_sentiments)
print(nb_tfidf)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LogisticRegression(C=1, max_iter=500, random_state=42)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LogisticRegression(C=1, max_iter=500, random_state=42)


#####Dự đoán trên tập test sử dụng mô hình đã huấn luyện

In [ ]:
lr_bow_predict = lr_bow.predict(cv_test_reviews)
lr_tfidf_predict = lr_tfidf.predict(tv_test_reviews)
nb_bow_predict = nb_bow.predict(cv_test_reviews)
nb_tfidf_predict = nb_tfidf.predict(tv_test_reviews)

#####Đánh giá mô hình

######Accuracy

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
#bow
lr_bow_score = accuracy_score(test_sentiments, lr_bow_predict)
print("lr_bow_score: ", lr_bow_score)
nb_bow_score = accuracy_score(test_sentiments, nb_bow_predict)
print("nb_bow_score: ", nb_bow_score)

#tf-idf
lr_tfidf_score=accuracy_score(test_sentiments,lr_tfidf_predict)
print("lr_tfidf_score :",lr_tfidf_score)
nb_tfidf_score=accuracy_score(test_sentiments,nb_tfidf_predict)
print("nb_tfidf_score :",nb_tfidf_score)

lr_bow_score:  0.9061
lr_tfidf_score : 0.9039


######Classification Report

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
lr_bow_report = classification_report(test_sentiments,
                                      lr_bow_predict,
                                      target_names = ['Positive', 'Negative'])
print("lr_bow_report: ", lr_bow_report)

nb_bow_report = classification_report(test_sentiments,
                                      nb_bow_predict,
                                      target_names = ['Positive', 'Negative'])
print("nb_bow_report: ", nb_bow_report)

lr_tfidf_report = classification_report(test_sentiments,
                                        lr_tfidf_predict,
                                        target_names = ['Positive', 'Negative'])
print("lr_tfidf_report: ", lr_tfidf_report)

nb_tfidf_report = classification_report(test_sentiments,
                                        nb_tfidf_predict,
                                        target_names = ['Positive', 'Negative'])
print("nb_tfidf_report: ", nb_tfidf_report)


lr_bow_report:                precision    recall  f1-score   support

    Positive       0.91      0.90      0.91      4993
    Negative       0.90      0.91      0.91      5007

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000

lr_tfidf_report:                precision    recall  f1-score   support

    Positive       0.91      0.90      0.90      4993
    Negative       0.90      0.91      0.90      5007

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



######Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
lr_bow_cm = confusion_matrix(test_sentiments, lr_bow_predict, labels=[1,0])
print("lr_bow_cm:\n", lr_bow_cm)

nb_bow_cm = confusion_matrix(test_sentiments, nb_bow_predict, labels=[1,0])
print("nb_bow_cm:\n", nb_bow_cm)

lr_tfidf_cm = confusion_matrix(test_sentiments, lr_tfidf_predict, labels=[1,0])
print("lr_tfidf_cm:\n", lr_tfidf_cm)

nb_tfidf_cm = confusion_matrix(test_sentiments, nb_tfidf_predict, labels=[1,0])
print("nb_tfidf_cm:\n", nb_tfidf_cm)

lr_bow_cm:
 [[4554  453]
 [ 486 4507]]
lr_tfidf_cm:
 [[4560  447]
 [ 514 4479]]


###Xuất file pkl và demo trên UI Streamlit

####Xuất file

In [ ]:
import os
import joblib

os.makedirs('pkl', exist_ok=True)

# Lưu bộ vectorizer BOW và model Logistic Regression tương ứng
joblib.dump(lr_bow, 'pkl/lr_bow_model.pkl')
joblib.dump(cv, 'pkl/bow_vectorizer.pkl')

# Lưu bộ vectorizer TF-IDF và model Logistic Regression tương ứng
joblib.dump(lr_tfidf, 'pkl/lr_tfidf_model.pkl')
joblib.dump(tv, 'pkl/tfidf_vectorizer.pkl')

# Lưu bộ vectorizer BOW và model Multinomial Naive Bayes tương ứng
joblib.dump(nb_bow, 'pkl/nb_bow_model.pkl')

# Lưu bộ vectorizer TF-IDF và model Multinomial Naive Bayes tương ứng
joblib.dump(nb_tfidf, 'pkl/nb_tfidf_model.pkl')


['tfidf_vectorizer.pkl']

####Xuất file vào UI streamlit

In [ ]:
!pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 56.2 MB/s eta 0:00:00


In [ ]:
!curl https://loca.lt/mytunnelpassword

8.229.170.247

In [ ]:
%%writefile app.py
import streamlit as st
import joblib

# Load lại từ file .pkl
model = joblib.load('lr_model_tfidf.pkl')
vectorizer = joblib.load('tfidf_vectorizer.pkl')

# Giả sử có dữ liệu mới từ người dùng
user_input = "I love this film so much!"

# BƯỚC QUAN TRỌNG: Dùng vectorizer để transform text mới
# Lưu ý: Chỉ dùng .transform(), không dùng .fit_transform()
input_vector = vectorizer.transform([user_input])

# Dự đoán
prediction = model.predict(input_vector)
st.write(f"Kết quả: {prediction[0]}")

Overwriting app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸

⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧your url is: https://late-parks-write.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.229.170.247:8501

  Stopping...
^C
